## 1 - Loading Processed Data

In this step, we load processed video dataset from the previously notebooks


In [1]:
import pandas as pd
import os

In [2]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fei_images_final.csv"
fei_imgs = pd.read_csv(load_path)

print(f"{len(fei_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(fei_imgs.sample(5))

load_path = "./processed_images/celeb_frames.csv"
celeb_imgs = pd.read_csv(load_path)

print(f"{len(celeb_imgs)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_imgs.sample(5))

--- LOADING PROCESSED DATA ---
5198 images loaded!


,path,label,split,dataset,method,target,source
3012,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_137-11_136-11_C16_B30_W30_PA16_PM00_F00_ssd.png,1,train,FEI,C16,137,136
2796,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/val/fake/M_156-11_125-11_C05_B30_W30_PA05_PM00_F00_ssd.png,1,val,FEI,C05,156,125
98,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_13-11_16-11_C15_B30_W30_PA15_PM00_F00_ssd.png,1,train,FEI,C15,13,16
834,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/fake/M_46-11_64-11_C15_B30_W30_PA15_PM00_F00_ssd.png,1,train,FEI,C15,46,64
5070,/Users/luciapola/Desktop/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework/Datasets/FEI_MORPHV2_DATASET/train/original/129-11_ssd.jpg,0,train,FEI,original,129,0


162255 images loaded!


,path,label,split,dataset,method,target,source
159067,CELEBDFV3_DATASET/test/fake/id35_id32_HyperReenact_id32_id35_0002_f1_ssd.jpg,1,test,Celeb-DF-v3,HyperReenact,id35,id32
149447,CELEBDFV3_DATASET/train/fake/id38_id23_TPSMM_id23_id38_0001_f2_ssd.jpg,1,train,Celeb-DF-v3,TPSMM,id38,id23
89767,CELEBDFV3_DATASET/val/fake/id31_id21_GHOST_id21_id31_0008_f1_ssd.jpg,1,val,Celeb-DF-v3,GHOST,id31,id21
140828,CELEBDFV3_DATASET/val/fake/id9_id35_MCNET_id35_id9_0002_f2_ssd.jpg,1,val,Celeb-DF-v3,MCNET,id9,id35
129162,CELEBDFV3_DATASET/test/fake/id35_id32_LIA_id32_id35_0005_f0_ssd.jpg,1,test,Celeb-DF-v3,LIA,id35,id32


## 2 - Master Dataset Compilation & Data Export

In this final preprocessing step, we consolidate our distinct datasets into standardized structures:

1. **Format Standardization:** We unify the column structure (`path`, `label`, `split`, `dataset`, `method`) across all dataframes and explicitly cast the labels into standard integers (`0` for Real, `1` for Fake).
2. **Master Dataset Assembly (FEI + Celeb-DF++):** We concatenate the FEI and Celeb-DF++ dataframes into a single, shuffled dataset. Each dataset keeps its own identity-based **Train**, **Validation** and **Test** split (computed in the respective preprocessing notebooks), so both datasets contribute to training, validation and testing. This `master_df` is exported as `master_dataset.csv`.
3. **No External Holdout:** Unlike the earlier Benchmark-phase pipeline (FF++-based), Celeb-DF++ is **not** held out here as a separate external test set — it is merged into the same train/val/test structure as FEI to increase training diversity. Cross-dataset generalization is therefore not measured by this master dataset; a dedicated external holdout would need to be reintroduced if that evaluation is required again.
4. **Final Audit:** We output grouped distribution tables and random samples to verify the final dataset composition, split proportions, and label balancing.

In [3]:
# Prepare FEI
df_fei = fei_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_fei['dataset'] = 'FEI'
df_fei['label'] = df_fei['label'].replace({'original': 0, 'fake': 1}).astype(int)

# Prepare CELEB
df_celeb = celeb_imgs[['path', 'label', 'split', 'method', 'target', 'source']].copy()
df_celeb['dataset'] = 'Celeb-DF'
df_celeb['label'] = df_celeb['label'].astype(int)

# Merge
master_df = pd.concat([df_fei, df_celeb], ignore_index=True)
master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
master_df.to_csv("master_dataset.csv", index=False)

print("Merge completed! File saved as 'master_dataset.csv'.")

print("\n--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---")
distribution_master = master_df.groupby(['dataset', 'method', 'split', 'label']).size().unstack(fill_value=0)

if len(distribution_master.columns) == 2:
    distribution_master.columns = ['0 (Real)', '1 (Fake)']
display(distribution_master)

print("Master Dataset Sample:")
display(master_df.sample(5))

Merge completed! File saved as 'master_dataset.csv'.

--- MASTER DATASET DISTRIBUTION (FEI + CELEB) ---


0 (Real)  1 (Fake)
dataset  method    split                    
Celeb-DF AniTalker test          0      1344
                   train         0      5739
                   val           0      1767
         BlendFace test          0      1146
                   train         0      3243
...                            ...       ...
FEI      C16       train         0       422
                   val           0       158
         original  test         54         0
                   train        92         0
                   val          54         0

[93 rows x 2 columns]

Master Dataset Sample:


,path,label,split,method,target,source,dataset
103398,CELEBDFV3_DATASET/train/fake/id41_id45_InSwapper_id45_id41_0003_f2_ssd.jpg,1,train,InSwapper,id41,id45,Celeb-DF
12249,CELEBDFV3_DATASET/train/fake/id22_id20_MCNET_id20_id22_0005_f0_ssd.jpg,1,train,MCNET,id22,id20,Celeb-DF
112701,CELEBDFV3_DATASET/train/fake/id06692_id41_EchoMimic_id41_0001_test_id06692_U7dRm7NKY-c_f1_ssd.jpg,1,train,EchoMimic,id06692,id41,Celeb-DF
143679,CELEBDFV3_DATASET/train/fake/id07663_id49_FLOAT_id49_0002_test_id07663_BwoN9bgCFWQ_f1_ssd.jpg,1,train,FLOAT,id07663,id49,Celeb-DF
128407,CELEBDFV3_DATASET/train/fake/id02057_id50_EDTalk_id50_0000_test_id02057_VCXnx-ozS8c_f0_ssd.jpg,1,train,EDTalk,id02057,id50,Celeb-DF


## 3 - Saving Data & Exporting Archive

To conclude this notebook, we first save our fully cleaned and processed DataFrames to local CSV files (e.g., `master_dataset.csv`). This ensures our prepared metadata is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the intensive preprocessing and extraction steps.

Finally, we package the entire structured dataset into a single, highly portable archive (`deepfake_dataset.zip`). This makes it easy to download, store, or transfer the data for model training.

**Contents of the final archive:**
* `FEI_MORPHV2_DATASET/`: The processed and split FEI dataset frames.
* `CELEBDFV3_DATASET/`: The processed and split Celeb-DF dataset frames.
* `master_dataset.csv`: The unified metadata for the training, validation, and test sets.

*(Note: We use the `-r` flag to include all subdirectories recursively and the `-q` flag to run the compression quietly, keeping the notebook output clean).*

In [4]:
print("--- SAVING DATAFRAME ---")

save_path_master = "master_dataset.csv"

master_df.to_csv(save_path_master, index=False)

print(f"Master Data successfully saved to: {save_path_master}")

--- SAVING DATAFRAME ---
Master Data successfully saved to: master_dataset.csv


In [5]:
!zip -rq deepfake_dataset.zip  FEI_MORPHV2_DATASET CELEBDFV3_DATASET master_dataset.csv